<a href="https://colab.research.google.com/github/AlekhyaGangopadhyay/IEM-IIT_HCI_Project/blob/main/chebyshev_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# Define the base path to your 'Synthetic Dataset' folder
# You might need to adjust this path based on where it's located in your Google Drive
base_path = '/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data'

# Dictionary to store file counts for each leaf folder
leaf_folder_file_counts = {}

# Traverse the directory tree
for root, dirs, files in os.walk(base_path):
    # Check if the current directory is a leaf folder (i.e., contains no subdirectories)
    if not dirs:
        # Count only files (excluding directories themselves if any are listed in 'files')
        file_count = len([f for f in files if os.path.isfile(os.path.join(root, f))])
        leaf_folder_file_counts[root] = file_count

# Print the results
print("File counts in leaf folders:")
for folder, count in leaf_folder_file_counts.items():
    print(f"  {folder}: {count} files")


File counts in leaf folders:
  /content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Right: 2096 files
  /content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Left: 2106 files
  /content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Forward: 2106 files
  /content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Backward: 2106 files


In [ ]:
# ============================================================
# CHEBYSHEV EEG FILTERING
# USER-DEFINED INPUT + OUTPUT PATH
# ============================================================

# INPUT:
# Any folder containing .xlsx EEG files
#
# OUTPUT:
# Filtered files saved in user-defined folder
#
# OUTPUT FILE NAME:
# chebyshev_originalfilename.xlsx
#
# ============================================================

!pip install openpyxl scipy -q

import os
import numpy as np
import pandas as pd

from scipy.signal import cheby1
from scipy.signal import sosfiltfilt

# ============================================================
# USER INPUT PATH
# ============================================================

INPUT_FOLDER ="/content/drive/My Drive/Human Computer Interface (HCI)/Synthetic Data/Subject 1/Right/Right"

# ============================================================
# USER OUTPUT PATH
# ============================================================

OUTPUT_FOLDER = "/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Right"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# EEG FILTER PARAMETERS
# ============================================================

FS = 250

LOWCUT = 8
HIGHCUT = 30

ORDER = 6

RP = 0.3

# ============================================================
# IMPROVED CHEBYSHEV FILTER
# ============================================================

def chebyshev_eeg_filter(
    data,
    fs=250,
    lowcut=8,
    highcut=30,
    order=6,
    rp=0.3
):

    """
    EEG Bandpass Filter
    -------------------
    - Chebyshev Type-I
    - Stable SOS filtering
    - Zero-phase filtering

    Input shape:
    [samples, channels]
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    # ========================================================
    # STABLE SECOND-ORDER SECTION FILTER
    # ========================================================

    sos = cheby1(
        N=order,
        rp=rp,
        Wn=[low, high],
        btype='bandpass',
        output='sos'
    )

    # ========================================================
    # ZERO-PHASE FILTERING
    # ========================================================

    filtered = sosfiltfilt(
        sos,
        data,
        axis=0
    )

    return filtered

# ============================================================
# GET XLSX FILES
# ============================================================

files_list = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".xlsx")
])

print("\n================================================")
print(f"FILES FOUND: {len(files_list)}")
print("================================================\n")

# ============================================================
# PROCESS FILES
# ============================================================

processed = 0

for file_name in files_list:

    try:

        file_path = os.path.join(
            INPUT_FOLDER,
            file_name
        )

        print(f"Processing: {file_name}")

        # ====================================================
        # LOAD EEG FILE
        # ====================================================

        df = pd.read_excel(
            file_path,
            engine='openpyxl'
        )

        data = df.values.astype(np.float32)

        columns = list(df.columns)

        # ====================================================
        # APPLY FILTER
        # ====================================================

        filtered_data = chebyshev_eeg_filter(
            data=data,
            fs=FS,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            order=ORDER,
            rp=RP
        )

        # ====================================================
        # SAVE FILTERED FILE
        # ====================================================

        output_name = (
            f"chebyshev_{file_name}"
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            output_name
        )

        pd.DataFrame(
            filtered_data,
            columns=columns
        ).to_excel(
            output_path,
            index=False,
            engine='openpyxl'
        )

        processed += 1

        print(f"Saved: {output_name}\n")

    except Exception as e:

        print(f"Error in file: {file_name}")
        print(e)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("CHEBYSHEV FILTERING COMPLETED")
print(f"TOTAL FILES PROCESSED: {processed}")
print("================================================")


FILES FOUND: 1043

Processing: ARROW_Right.xlsx
Saved: chebyshev_ARROW_Right.xlsx

Processing: ARROW_Right_synthetic_1.xlsx
Saved: chebyshev_ARROW_Right_synthetic_1.xlsx

Processing: ARROW_Right_synthetic_10.xlsx
Saved: chebyshev_ARROW_Right_synthetic_10.xlsx

Processing: ARROW_Right_synthetic_100.xlsx
Saved: chebyshev_ARROW_Right_synthetic_100.xlsx

Processing: ARROW_Right_synthetic_11.xlsx
Saved: chebyshev_ARROW_Right_synthetic_11.xlsx

Processing: ARROW_Right_synthetic_12.xlsx
Saved: chebyshev_ARROW_Right_synthetic_12.xlsx

Processing: ARROW_Right_synthetic_13.xlsx
Saved: chebyshev_ARROW_Right_synthetic_13.xlsx

Processing: ARROW_Right_synthetic_14.xlsx
Saved: chebyshev_ARROW_Right_synthetic_14.xlsx

Processing: ARROW_Right_synthetic_15.xlsx
Saved: chebyshev_ARROW_Right_synthetic_15.xlsx

Processing: ARROW_Right_synthetic_16.xlsx
Saved: chebyshev_ARROW_Right_synthetic_16.xlsx

Processing: ARROW_Right_synthetic_17.xlsx
Saved: chebyshev_ARROW_Right_synthetic_17.xlsx

Processing: ARRO

In [ ]:
# ============================================================
# CHEBYSHEV EEG FILTERING
# USER-DEFINED INPUT + OUTPUT PATH
# ============================================================

# INPUT:
# Any folder containing .xlsx EEG files
#
# OUTPUT:
# Filtered files saved in user-defined folder
#
# OUTPUT FILE NAME:
# chebyshev_originalfilename.xlsx
#
# ============================================================

!pip install openpyxl scipy -q

import os
import numpy as np
import pandas as pd

from scipy.signal import cheby1
from scipy.signal import sosfiltfilt

# ============================================================
# USER INPUT PATH
# ============================================================

INPUT_FOLDER ="/content/drive/My Drive/Human Computer Interface (HCI)/Synthetic Data/Subject 1/Left"

# ============================================================
# USER OUTPUT PATH
# ============================================================

OUTPUT_FOLDER = "/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Left"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# EEG FILTER PARAMETERS
# ============================================================

FS = 250

LOWCUT = 8
HIGHCUT = 30

ORDER = 6

RP = 0.3

# ============================================================
# IMPROVED CHEBYSHEV FILTER
# ============================================================

def chebyshev_eeg_filter(
    data,
    fs=250,
    lowcut=8,
    highcut=30,
    order=6,
    rp=0.3
):

    """
    EEG Bandpass Filter
    -------------------
    - Chebyshev Type-I
    - Stable SOS filtering
    - Zero-phase filtering

    Input shape:
    [samples, channels]
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    # ========================================================
    # STABLE SECOND-ORDER SECTION FILTER
    # ========================================================

    sos = cheby1(
        N=order,
        rp=rp,
        Wn=[low, high],
        btype='bandpass',
        output='sos'
    )

    # ========================================================
    # ZERO-PHASE FILTERING
    # ========================================================

    filtered = sosfiltfilt(
        sos,
        data,
        axis=0
    )

    return filtered

# ============================================================
# GET XLSX FILES
# ============================================================

files_list = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".xlsx")
])

print("\n================================================")
print(f"FILES FOUND: {len(files_list)}")
print("================================================\n")

# ============================================================
# PROCESS FILES
# ============================================================

processed = 0

for file_name in files_list:

    try:

        file_path = os.path.join(
            INPUT_FOLDER,
            file_name
        )

        print(f"Processing: {file_name}")

        # ====================================================
        # LOAD EEG FILE
        # ====================================================

        df = pd.read_excel(
            file_path,
            engine='openpyxl'
        )

        data = df.values.astype(np.float32)

        columns = list(df.columns)

        # ====================================================
        # APPLY FILTER
        # ====================================================

        filtered_data = chebyshev_eeg_filter(
            data=data,
            fs=FS,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            order=ORDER,
            rp=RP
        )

        # ====================================================
        # SAVE FILTERED FILE
        # ====================================================

        output_name = (
            f"chebyshev_{file_name}"
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            output_name
        )

        pd.DataFrame(
            filtered_data,
            columns=columns
        ).to_excel(
            output_path,
            index=False,
            engine='openpyxl'
        )

        processed += 1

        print(f"Saved: {output_name}\n")

    except Exception as e:

        print(f"Error in file: {file_name}")
        print(e)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("CHEBYSHEV FILTERING COMPLETED")
print(f"TOTAL FILES PROCESSED: {processed}")
print("================================================")


FILES FOUND: 1053

Processing: ARROW_Left.xlsx
Saved: chebyshev_ARROW_Left.xlsx

Processing: ARROW_Left_Subject_1_synthetic_1.xlsx
Saved: chebyshev_ARROW_Left_Subject_1_synthetic_1.xlsx

Processing: ARROW_Left_Subject_1_synthetic_10.xlsx
Saved: chebyshev_ARROW_Left_Subject_1_synthetic_10.xlsx

Processing: ARROW_Left_Subject_1_synthetic_100.xlsx
Saved: chebyshev_ARROW_Left_Subject_1_synthetic_100.xlsx

Processing: ARROW_Left_Subject_1_synthetic_11.xlsx
Saved: chebyshev_ARROW_Left_Subject_1_synthetic_11.xlsx

Processing: ARROW_Left_Subject_1_synthetic_12.xlsx
Saved: chebyshev_ARROW_Left_Subject_1_synthetic_12.xlsx

Processing: ARROW_Left_Subject_1_synthetic_13.xlsx
Saved: chebyshev_ARROW_Left_Subject_1_synthetic_13.xlsx

Processing: ARROW_Left_Subject_1_synthetic_14.xlsx
Saved: chebyshev_ARROW_Left_Subject_1_synthetic_14.xlsx

Processing: ARROW_Left_Subject_1_synthetic_15.xlsx
Saved: chebyshev_ARROW_Left_Subject_1_synthetic_15.xlsx

Processing: ARROW_Left_Subject_1_synthetic_16.xlsx
Sav

In [ ]:
# ============================================================
# CHEBYSHEV EEG FILTERING
# USER-DEFINED INPUT + OUTPUT PATH
# ============================================================

# INPUT:
# Any folder containing .xlsx EEG files
#
# OUTPUT:
# Filtered files saved in user-defined folder
#
# OUTPUT FILE NAME:
# chebyshev_originalfilename.xlsx
#
# ============================================================

!pip install openpyxl scipy -q

import os
import numpy as np
import pandas as pd

from scipy.signal import cheby1
from scipy.signal import sosfiltfilt

# ============================================================
# USER INPUT PATH
# ============================================================

INPUT_FOLDER ="/content/drive/My Drive/Human Computer Interface (HCI)/Synthetic Data/Subject 1/Forward"

# ============================================================
# USER OUTPUT PATH
# ============================================================

OUTPUT_FOLDER = "/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Forward"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# EEG FILTER PARAMETERS
# ============================================================

FS = 250

LOWCUT = 8
HIGHCUT = 30

ORDER = 6

RP = 0.3

# ============================================================
# IMPROVED CHEBYSHEV FILTER
# ============================================================

def chebyshev_eeg_filter(
    data,
    fs=250,
    lowcut=8,
    highcut=30,
    order=6,
    rp=0.3
):

    """
    EEG Bandpass Filter
    -------------------
    - Chebyshev Type-I
    - Stable SOS filtering
    - Zero-phase filtering

    Input shape:
    [samples, channels]
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    # ========================================================
    # STABLE SECOND-ORDER SECTION FILTER
    # ========================================================

    sos = cheby1(
        N=order,
        rp=rp,
        Wn=[low, high],
        btype='bandpass',
        output='sos'
    )

    # ========================================================
    # ZERO-PHASE FILTERING
    # ========================================================

    filtered = sosfiltfilt(
        sos,
        data,
        axis=0
    )

    return filtered

# ============================================================
# GET XLSX FILES
# ============================================================

files_list = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".xlsx")
])

print("\n================================================")
print(f"FILES FOUND: {len(files_list)}")
print("================================================\n")

# ============================================================
# PROCESS FILES
# ============================================================

processed = 0

for file_name in files_list:

    try:

        file_path = os.path.join(
            INPUT_FOLDER,
            file_name
        )

        print(f"Processing: {file_name}")

        # ====================================================
        # LOAD EEG FILE
        # ====================================================

        df = pd.read_excel(
            file_path,
            engine='openpyxl'
        )

        data = df.values.astype(np.float32)

        columns = list(df.columns)

        # ====================================================
        # APPLY FILTER
        # ====================================================

        filtered_data = chebyshev_eeg_filter(
            data=data,
            fs=FS,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            order=ORDER,
            rp=RP
        )

        # ====================================================
        # SAVE FILTERED FILE
        # ====================================================

        output_name = (
            f"chebyshev_{file_name}"
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            output_name
        )

        pd.DataFrame(
            filtered_data,
            columns=columns
        ).to_excel(
            output_path,
            index=False,
            engine='openpyxl'
        )

        processed += 1

        print(f"Saved: {output_name}\n")

    except Exception as e:

        print(f"Error in file: {file_name}")
        print(e)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("CHEBYSHEV FILTERING COMPLETED")
print(f"TOTAL FILES PROCESSED: {processed}")
print("================================================")


FILES FOUND: 1053

Processing: ARROW_Forward.xlsx
Saved: chebyshev_ARROW_Forward.xlsx

Processing: ARROW_Forward_Subject_1_synthetic_1.xlsx
Saved: chebyshev_ARROW_Forward_Subject_1_synthetic_1.xlsx

Processing: ARROW_Forward_Subject_1_synthetic_10.xlsx
Saved: chebyshev_ARROW_Forward_Subject_1_synthetic_10.xlsx

Processing: ARROW_Forward_Subject_1_synthetic_100.xlsx
Saved: chebyshev_ARROW_Forward_Subject_1_synthetic_100.xlsx

Processing: ARROW_Forward_Subject_1_synthetic_11.xlsx
Saved: chebyshev_ARROW_Forward_Subject_1_synthetic_11.xlsx

Processing: ARROW_Forward_Subject_1_synthetic_12.xlsx
Saved: chebyshev_ARROW_Forward_Subject_1_synthetic_12.xlsx

Processing: ARROW_Forward_Subject_1_synthetic_13.xlsx
Saved: chebyshev_ARROW_Forward_Subject_1_synthetic_13.xlsx

Processing: ARROW_Forward_Subject_1_synthetic_14.xlsx
Saved: chebyshev_ARROW_Forward_Subject_1_synthetic_14.xlsx

Processing: ARROW_Forward_Subject_1_synthetic_15.xlsx
Saved: chebyshev_ARROW_Forward_Subject_1_synthetic_15.xlsx



In [ ]:
# ============================================================
# CHEBYSHEV EEG FILTERING
# USER-DEFINED INPUT + OUTPUT PATH
# ============================================================

# INPUT:
# Any folder containing .xlsx EEG files
#
# OUTPUT:
# Filtered files saved in user-defined folder
#
# OUTPUT FILE NAME:
# chebyshev_originalfilename.xlsx
#
# ============================================================

!pip install openpyxl scipy -q

import os
import numpy as np
import pandas as pd

from scipy.signal import cheby1
from scipy.signal import sosfiltfilt

# ============================================================
# USER INPUT PATH
# ============================================================

INPUT_FOLDER ="/content/drive/My Drive/Human Computer Interface (HCI)/Synthetic Data/Subject 1/Backward"

# ============================================================
# USER OUTPUT PATH
# ============================================================

OUTPUT_FOLDER = "/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Backward"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# EEG FILTER PARAMETERS
# ============================================================

FS = 250

LOWCUT = 8
HIGHCUT = 30

ORDER = 6

RP = 0.3

# ============================================================
# IMPROVED CHEBYSHEV FILTER
# ============================================================

def chebyshev_eeg_filter(
    data,
    fs=250,
    lowcut=8,
    highcut=30,
    order=6,
    rp=0.3
):

    """
    EEG Bandpass Filter
    -------------------
    - Chebyshev Type-I
    - Stable SOS filtering
    - Zero-phase filtering

    Input shape:
    [samples, channels]
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    # ========================================================
    # STABLE SECOND-ORDER SECTION FILTER
    # ========================================================

    sos = cheby1(
        N=order,
        rp=rp,
        Wn=[low, high],
        btype='bandpass',
        output='sos'
    )

    # ========================================================
    # ZERO-PHASE FILTERING
    # ========================================================

    filtered = sosfiltfilt(
        sos,
        data,
        axis=0
    )

    return filtered

# ============================================================
# GET XLSX FILES
# ============================================================

files_list = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".xlsx")
])

print("\n================================================")
print(f"FILES FOUND: {len(files_list)}")
print("================================================\n")

# ============================================================
# PROCESS FILES
# ============================================================

processed = 0

for file_name in files_list:

    try:

        file_path = os.path.join(
            INPUT_FOLDER,
            file_name
        )

        print(f"Processing: {file_name}")

        # ====================================================
        # LOAD EEG FILE
        # ====================================================

        df = pd.read_excel(
            file_path,
            engine='openpyxl'
        )

        data = df.values.astype(np.float32)

        columns = list(df.columns)

        # ====================================================
        # APPLY FILTER
        # ====================================================

        filtered_data = chebyshev_eeg_filter(
            data=data,
            fs=FS,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            order=ORDER,
            rp=RP
        )

        # ====================================================
        # SAVE FILTERED FILE
        # ====================================================

        output_name = (
            f"chebyshev_{file_name}"
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            output_name
        )

        pd.DataFrame(
            filtered_data,
            columns=columns
        ).to_excel(
            output_path,
            index=False,
            engine='openpyxl'
        )

        processed += 1

        print(f"Saved: {output_name}\n")

    except Exception as e:

        print(f"Error in file: {file_name}")
        print(e)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("CHEBYSHEV FILTERING COMPLETED")
print(f"TOTAL FILES PROCESSED: {processed}")
print("================================================")


FILES FOUND: 1053

Processing: ARROW_Backward.xlsx
Saved: chebyshev_ARROW_Backward.xlsx

Processing: ARROW_Backward_Subject_1_synthetic_1.xlsx
Saved: chebyshev_ARROW_Backward_Subject_1_synthetic_1.xlsx

Processing: ARROW_Backward_Subject_1_synthetic_10.xlsx
Saved: chebyshev_ARROW_Backward_Subject_1_synthetic_10.xlsx

Processing: ARROW_Backward_Subject_1_synthetic_100.xlsx
Saved: chebyshev_ARROW_Backward_Subject_1_synthetic_100.xlsx

Processing: ARROW_Backward_Subject_1_synthetic_11.xlsx
Saved: chebyshev_ARROW_Backward_Subject_1_synthetic_11.xlsx

Processing: ARROW_Backward_Subject_1_synthetic_12.xlsx
Saved: chebyshev_ARROW_Backward_Subject_1_synthetic_12.xlsx

Processing: ARROW_Backward_Subject_1_synthetic_13.xlsx
Saved: chebyshev_ARROW_Backward_Subject_1_synthetic_13.xlsx

Processing: ARROW_Backward_Subject_1_synthetic_14.xlsx
Saved: chebyshev_ARROW_Backward_Subject_1_synthetic_14.xlsx

Processing: ARROW_Backward_Subject_1_synthetic_15.xlsx
Saved: chebyshev_ARROW_Backward_Subject_1_s

In [ ]:
# ============================================================
# CHEBYSHEV EEG FILTERING
# USER-DEFINED INPUT + OUTPUT PATH
# ============================================================

# INPUT:
# Any folder containing .xlsx EEG files
#
# OUTPUT:
# Filtered files saved in user-defined folder
#
# OUTPUT FILE NAME:
# chebyshev_originalfilename.xlsx
#
# ============================================================

!pip install openpyxl scipy -q

import os
import numpy as np
import pandas as pd

from scipy.signal import cheby1
from scipy.signal import sosfiltfilt

# ============================================================
# USER INPUT PATH
# ============================================================

INPUT_FOLDER ="/content/drive/My Drive/Human Computer Interface (HCI)/Synthetic Data/Subject 2/Right"

# ============================================================
# USER OUTPUT PATH
# ============================================================

OUTPUT_FOLDER = "/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Right"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# EEG FILTER PARAMETERS
# ============================================================

FS = 250

LOWCUT = 8
HIGHCUT = 30

ORDER = 6

RP = 0.3

# ============================================================
# IMPROVED CHEBYSHEV FILTER
# ============================================================

def chebyshev_eeg_filter(
    data,
    fs=250,
    lowcut=8,
    highcut=30,
    order=6,
    rp=0.3
):

    """
    EEG Bandpass Filter
    -------------------
    - Chebyshev Type-I
    - Stable SOS filtering
    - Zero-phase filtering

    Input shape:
    [samples, channels]
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    # ========================================================
    # STABLE SECOND-ORDER SECTION FILTER
    # ========================================================

    sos = cheby1(
        N=order,
        rp=rp,
        Wn=[low, high],
        btype='bandpass',
        output='sos'
    )

    # ========================================================
    # ZERO-PHASE FILTERING
    # ========================================================

    filtered = sosfiltfilt(
        sos,
        data,
        axis=0
    )

    return filtered

# ============================================================
# GET XLSX FILES
# ============================================================

files_list = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".xlsx")
])

print("\n================================================")
print(f"FILES FOUND: {len(files_list)}")
print("================================================\n")

# ============================================================
# PROCESS FILES
# ============================================================

processed = 0

for file_name in files_list:

    try:

        file_path = os.path.join(
            INPUT_FOLDER,
            file_name
        )

        print(f"Processing: {file_name}")

        # ====================================================
        # LOAD EEG FILE
        # ====================================================

        df = pd.read_excel(
            file_path,
            engine='openpyxl'
        )

        data = df.values.astype(np.float32)

        columns = list(df.columns)

        # ====================================================
        # APPLY FILTER
        # ====================================================

        filtered_data = chebyshev_eeg_filter(
            data=data,
            fs=FS,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            order=ORDER,
            rp=RP
        )

        # ====================================================
        # SAVE FILTERED FILE
        # ====================================================

        output_name = (
            f"chebyshev_{file_name}"
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            output_name
        )

        pd.DataFrame(
            filtered_data,
            columns=columns
        ).to_excel(
            output_path,
            index=False,
            engine='openpyxl'
        )

        processed += 1

        print(f"Saved: {output_name}\n")

    except Exception as e:

        print(f"Error in file: {file_name}")
        print(e)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("CHEBYSHEV FILTERING COMPLETED")
print(f"TOTAL FILES PROCESSED: {processed}")
print("================================================")


FILES FOUND: 1053

Processing: Right ARROW2.xlsx
Saved: chebyshev_Right ARROW2.xlsx

Processing: Right ARROW2_Subject_2_synthetic_1.xlsx
Saved: chebyshev_Right ARROW2_Subject_2_synthetic_1.xlsx

Processing: Right ARROW2_Subject_2_synthetic_10.xlsx
Saved: chebyshev_Right ARROW2_Subject_2_synthetic_10.xlsx

Processing: Right ARROW2_Subject_2_synthetic_100.xlsx
Saved: chebyshev_Right ARROW2_Subject_2_synthetic_100.xlsx

Processing: Right ARROW2_Subject_2_synthetic_11.xlsx
Saved: chebyshev_Right ARROW2_Subject_2_synthetic_11.xlsx

Processing: Right ARROW2_Subject_2_synthetic_12.xlsx
Saved: chebyshev_Right ARROW2_Subject_2_synthetic_12.xlsx

Processing: Right ARROW2_Subject_2_synthetic_13.xlsx
Saved: chebyshev_Right ARROW2_Subject_2_synthetic_13.xlsx

Processing: Right ARROW2_Subject_2_synthetic_14.xlsx
Saved: chebyshev_Right ARROW2_Subject_2_synthetic_14.xlsx

Processing: Right ARROW2_Subject_2_synthetic_15.xlsx
Saved: chebyshev_Right ARROW2_Subject_2_synthetic_15.xlsx

Processing: Right 

In [ ]:
# ============================================================
# CHEBYSHEV EEG FILTERING
# USER-DEFINED INPUT + OUTPUT PATH
# ============================================================

# INPUT:
# Any folder containing .xlsx EEG files
#
# OUTPUT:
# Filtered files saved in user-defined folder
#
# OUTPUT FILE NAME:
# chebyshev_originalfilename.xlsx
#
# ============================================================

!pip install openpyxl scipy -q

import os
import numpy as np
import pandas as pd

from scipy.signal import cheby1
from scipy.signal import sosfiltfilt

# ============================================================
# USER INPUT PATH
# ============================================================

INPUT_FOLDER ="/content/drive/My Drive/Human Computer Interface (HCI)/Synthetic Data/Subject 2/Left"

# ============================================================
# USER OUTPUT PATH
# ============================================================

OUTPUT_FOLDER = "/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Left"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# EEG FILTER PARAMETERS
# ============================================================

FS = 250

LOWCUT = 8
HIGHCUT = 30

ORDER = 6

RP = 0.3

# ============================================================
# IMPROVED CHEBYSHEV FILTER
# ============================================================

def chebyshev_eeg_filter(
    data,
    fs=250,
    lowcut=8,
    highcut=30,
    order=6,
    rp=0.3
):

    """
    EEG Bandpass Filter
    -------------------
    - Chebyshev Type-I
    - Stable SOS filtering
    - Zero-phase filtering

    Input shape:
    [samples, channels]
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    # ========================================================
    # STABLE SECOND-ORDER SECTION FILTER
    # ========================================================

    sos = cheby1(
        N=order,
        rp=rp,
        Wn=[low, high],
        btype='bandpass',
        output='sos'
    )

    # ========================================================
    # ZERO-PHASE FILTERING
    # ========================================================

    filtered = sosfiltfilt(
        sos,
        data,
        axis=0
    )

    return filtered

# ============================================================
# GET XLSX FILES
# ============================================================

files_list = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".xlsx")
])

print("\n================================================")
print(f"FILES FOUND: {len(files_list)}")
print("================================================\n")

# ============================================================
# PROCESS FILES
# ============================================================

processed = 0

for file_name in files_list:

    try:

        file_path = os.path.join(
            INPUT_FOLDER,
            file_name
        )

        print(f"Processing: {file_name}")

        # ====================================================
        # LOAD EEG FILE
        # ====================================================

        df = pd.read_excel(
            file_path,
            engine='openpyxl'
        )

        data = df.values.astype(np.float32)

        columns = list(df.columns)

        # ====================================================
        # APPLY FILTER
        # ====================================================

        filtered_data = chebyshev_eeg_filter(
            data=data,
            fs=FS,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            order=ORDER,
            rp=RP
        )

        # ====================================================
        # SAVE FILTERED FILE
        # ====================================================

        output_name = (
            f"chebyshev_{file_name}"
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            output_name
        )

        pd.DataFrame(
            filtered_data,
            columns=columns
        ).to_excel(
            output_path,
            index=False,
            engine='openpyxl'
        )

        processed += 1

        print(f"Saved: {output_name}\n")

    except Exception as e:

        print(f"Error in file: {file_name}")
        print(e)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("CHEBYSHEV FILTERING COMPLETED")
print(f"TOTAL FILES PROCESSED: {processed}")
print("================================================")


FILES FOUND: 1053

Processing: ARROW_Left_2.xlsx
Saved: chebyshev_ARROW_Left_2.xlsx

Processing: ARROW_Left_2_Subject_2_synthetic_1.xlsx
Saved: chebyshev_ARROW_Left_2_Subject_2_synthetic_1.xlsx

Processing: ARROW_Left_2_Subject_2_synthetic_10.xlsx
Saved: chebyshev_ARROW_Left_2_Subject_2_synthetic_10.xlsx

Processing: ARROW_Left_2_Subject_2_synthetic_100.xlsx
Saved: chebyshev_ARROW_Left_2_Subject_2_synthetic_100.xlsx

Processing: ARROW_Left_2_Subject_2_synthetic_11.xlsx
Saved: chebyshev_ARROW_Left_2_Subject_2_synthetic_11.xlsx

Processing: ARROW_Left_2_Subject_2_synthetic_12.xlsx
Saved: chebyshev_ARROW_Left_2_Subject_2_synthetic_12.xlsx

Processing: ARROW_Left_2_Subject_2_synthetic_13.xlsx
Saved: chebyshev_ARROW_Left_2_Subject_2_synthetic_13.xlsx

Processing: ARROW_Left_2_Subject_2_synthetic_14.xlsx
Saved: chebyshev_ARROW_Left_2_Subject_2_synthetic_14.xlsx

Processing: ARROW_Left_2_Subject_2_synthetic_15.xlsx
Saved: chebyshev_ARROW_Left_2_Subject_2_synthetic_15.xlsx

Processing: ARROW_

In [ ]:
# ============================================================
# CHEBYSHEV EEG FILTERING
# USER-DEFINED INPUT + OUTPUT PATH
# ============================================================

# INPUT:
# Any folder containing .xlsx EEG files
#
# OUTPUT:
# Filtered files saved in user-defined folder
#
# OUTPUT FILE NAME:
# chebyshev_originalfilename.xlsx
#
# ============================================================

!pip install openpyxl scipy -q

import os
import numpy as np
import pandas as pd

from scipy.signal import cheby1
from scipy.signal import sosfiltfilt

# ============================================================
# USER INPUT PATH
# ============================================================

INPUT_FOLDER ="/content/drive/My Drive/Human Computer Interface (HCI)/Synthetic Data/Subject 2/Forward"

# ============================================================
# USER OUTPUT PATH
# ============================================================

OUTPUT_FOLDER = "/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Forward"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# EEG FILTER PARAMETERS
# ============================================================

FS = 250

LOWCUT = 8
HIGHCUT = 30

ORDER = 6

RP = 0.3

# ============================================================
# IMPROVED CHEBYSHEV FILTER
# ============================================================

def chebyshev_eeg_filter(
    data,
    fs=250,
    lowcut=8,
    highcut=30,
    order=6,
    rp=0.3
):

    """
    EEG Bandpass Filter
    -------------------
    - Chebyshev Type-I
    - Stable SOS filtering
    - Zero-phase filtering

    Input shape:
    [samples, channels]
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    # ========================================================
    # STABLE SECOND-ORDER SECTION FILTER
    # ========================================================

    sos = cheby1(
        N=order,
        rp=rp,
        Wn=[low, high],
        btype='bandpass',
        output='sos'
    )

    # ========================================================
    # ZERO-PHASE FILTERING
    # ========================================================

    filtered = sosfiltfilt(
        sos,
        data,
        axis=0
    )

    return filtered

# ============================================================
# GET XLSX FILES
# ============================================================

files_list = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".xlsx")
])

print("\n================================================")
print(f"FILES FOUND: {len(files_list)}")
print("================================================\n")

# ============================================================
# PROCESS FILES
# ============================================================

processed = 0

for file_name in files_list:

    try:

        file_path = os.path.join(
            INPUT_FOLDER,
            file_name
        )

        print(f"Processing: {file_name}")

        # ====================================================
        # LOAD EEG FILE
        # ====================================================

        df = pd.read_excel(
            file_path,
            engine='openpyxl'
        )

        data = df.values.astype(np.float32)

        columns = list(df.columns)

        # ====================================================
        # APPLY FILTER
        # ====================================================

        filtered_data = chebyshev_eeg_filter(
            data=data,
            fs=FS,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            order=ORDER,
            rp=RP
        )

        # ====================================================
        # SAVE FILTERED FILE
        # ====================================================

        output_name = (
            f"chebyshev_{file_name}"
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            output_name
        )

        pd.DataFrame(
            filtered_data,
            columns=columns
        ).to_excel(
            output_path,
            index=False,
            engine='openpyxl'
        )

        processed += 1

        print(f"Saved: {output_name}\n")

    except Exception as e:

        print(f"Error in file: {file_name}")
        print(e)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("CHEBYSHEV FILTERING COMPLETED")
print(f"TOTAL FILES PROCESSED: {processed}")
print("================================================")


FILES FOUND: 1053

Processing: ARROW_Forward_2.xlsx
Saved: chebyshev_ARROW_Forward_2.xlsx

Processing: ARROW_Forward_2_Subject_2_synthetic_1.xlsx
Saved: chebyshev_ARROW_Forward_2_Subject_2_synthetic_1.xlsx

Processing: ARROW_Forward_2_Subject_2_synthetic_10.xlsx
Saved: chebyshev_ARROW_Forward_2_Subject_2_synthetic_10.xlsx

Processing: ARROW_Forward_2_Subject_2_synthetic_100.xlsx
Saved: chebyshev_ARROW_Forward_2_Subject_2_synthetic_100.xlsx

Processing: ARROW_Forward_2_Subject_2_synthetic_11.xlsx
Saved: chebyshev_ARROW_Forward_2_Subject_2_synthetic_11.xlsx

Processing: ARROW_Forward_2_Subject_2_synthetic_12.xlsx
Saved: chebyshev_ARROW_Forward_2_Subject_2_synthetic_12.xlsx

Processing: ARROW_Forward_2_Subject_2_synthetic_13.xlsx
Saved: chebyshev_ARROW_Forward_2_Subject_2_synthetic_13.xlsx

Processing: ARROW_Forward_2_Subject_2_synthetic_14.xlsx
Saved: chebyshev_ARROW_Forward_2_Subject_2_synthetic_14.xlsx

Processing: ARROW_Forward_2_Subject_2_synthetic_15.xlsx
Saved: chebyshev_ARROW_For

In [ ]:
# ============================================================
# CHEBYSHEV EEG FILTERING
# USER-DEFINED INPUT + OUTPUT PATH
# ============================================================

# INPUT:
# Any folder containing .xlsx EEG files
#
# OUTPUT:
# Filtered files saved in user-defined folder
#
# OUTPUT FILE NAME:
# chebyshev_originalfilename.xlsx
#
# ============================================================

!pip install openpyxl scipy -q

import os
import numpy as np
import pandas as pd

from scipy.signal import cheby1
from scipy.signal import sosfiltfilt

# ============================================================
# USER INPUT PATH
# ============================================================

INPUT_FOLDER ="/content/drive/My Drive/Human Computer Interface (HCI)/Synthetic Data/Subject 2/Backward"

# ============================================================
# USER OUTPUT PATH
# ============================================================

OUTPUT_FOLDER = "/content/drive/My Drive/Human Computer Interface (HCI)/Chebyshev Filtered Data/Backward"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# EEG FILTER PARAMETERS
# ============================================================

FS = 250

LOWCUT = 8
HIGHCUT = 30

ORDER = 6

RP = 0.3

# ============================================================
# IMPROVED CHEBYSHEV FILTER
# ============================================================

def chebyshev_eeg_filter(
    data,
    fs=250,
    lowcut=8,
    highcut=30,
    order=6,
    rp=0.3
):

    """
    EEG Bandpass Filter
    -------------------
    - Chebyshev Type-I
    - Stable SOS filtering
    - Zero-phase filtering

    Input shape:
    [samples, channels]
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    # ========================================================
    # STABLE SECOND-ORDER SECTION FILTER
    # ========================================================

    sos = cheby1(
        N=order,
        rp=rp,
        Wn=[low, high],
        btype='bandpass',
        output='sos'
    )

    # ========================================================
    # ZERO-PHASE FILTERING
    # ========================================================

    filtered = sosfiltfilt(
        sos,
        data,
        axis=0
    )

    return filtered

# ============================================================
# GET XLSX FILES
# ============================================================

files_list = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".xlsx")
])

print("\n================================================")
print(f"FILES FOUND: {len(files_list)}")
print("================================================\n")

# ============================================================
# PROCESS FILES
# ============================================================

processed = 0

for file_name in files_list:

    try:

        file_path = os.path.join(
            INPUT_FOLDER,
            file_name
        )

        print(f"Processing: {file_name}")

        # ====================================================
        # LOAD EEG FILE
        # ====================================================

        df = pd.read_excel(
            file_path,
            engine='openpyxl'
        )

        data = df.values.astype(np.float32)

        columns = list(df.columns)

        # ====================================================
        # APPLY FILTER
        # ====================================================

        filtered_data = chebyshev_eeg_filter(
            data=data,
            fs=FS,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            order=ORDER,
            rp=RP
        )

        # ====================================================
        # SAVE FILTERED FILE
        # ====================================================

        output_name = (
            f"chebyshev_{file_name}"
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            output_name
        )

        pd.DataFrame(
            filtered_data,
            columns=columns
        ).to_excel(
            output_path,
            index=False,
            engine='openpyxl'
        )

        processed += 1

        print(f"Saved: {output_name}\n")

    except Exception as e:

        print(f"Error in file: {file_name}")
        print(e)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("CHEBYSHEV FILTERING COMPLETED")
print(f"TOTAL FILES PROCESSED: {processed}")
print("================================================")


FILES FOUND: 1053

Processing: ARROW_Backword_2.xlsx
Saved: chebyshev_ARROW_Backword_2.xlsx

Processing: ARROW_Backword_2_Subject_2_synthetic_1.xlsx
Saved: chebyshev_ARROW_Backword_2_Subject_2_synthetic_1.xlsx

Processing: ARROW_Backword_2_Subject_2_synthetic_10.xlsx
Saved: chebyshev_ARROW_Backword_2_Subject_2_synthetic_10.xlsx

Processing: ARROW_Backword_2_Subject_2_synthetic_100.xlsx
Saved: chebyshev_ARROW_Backword_2_Subject_2_synthetic_100.xlsx

Processing: ARROW_Backword_2_Subject_2_synthetic_11.xlsx
Saved: chebyshev_ARROW_Backword_2_Subject_2_synthetic_11.xlsx

Processing: ARROW_Backword_2_Subject_2_synthetic_12.xlsx
Saved: chebyshev_ARROW_Backword_2_Subject_2_synthetic_12.xlsx

Processing: ARROW_Backword_2_Subject_2_synthetic_13.xlsx
Saved: chebyshev_ARROW_Backword_2_Subject_2_synthetic_13.xlsx

Processing: ARROW_Backword_2_Subject_2_synthetic_14.xlsx
Saved: chebyshev_ARROW_Backword_2_Subject_2_synthetic_14.xlsx

Processing: ARROW_Backword_2_Subject_2_synthetic_15.xlsx
Saved: ch